In [138]:
import tensorflow as tf
import os, datetime
import tensorflow_recommenders as tfrs
from google.cloud import bigquery
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import ndcg_score
import pprint
from typing import Dict, Text
import logging
import faiss
from keras_tuner import HyperModel, HyperParameters
from keras_tuner.tuners import BayesianOptimization, Hyperband, RandomSearch
from keras_tuner import Objective
import optuna

import xgboost as xgb
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

/Users/dmmckinn/repos/movie_recommender/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [78]:
# Check for GPU availability and set memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logger.info(f"GPUs available: {len(gpus)}")
    except RuntimeError as e:
        logger.warning(e)
else:
    logger.info("No GPUs available.")

2025-01-24 10:02:59,652 - INFO - No GPUs available.


In [79]:
# Define the BigQuery table and project details
PROJECT_ID = 'oolola'
DATASET_ID = 'movie_data'
TABLE_ID   = 'preprocessed_data'
timestamp  = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
output_dir = 'gs://movie-data-1/trained-model'
# Function to load movies from BigQuery
def load_movies_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT title, genres
        FROM `{PROJECT_ID}.{DATASET_ID}.preprocessed_movies`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading movies from BigQuery: {e}")
        raise
# Function to load ratings from BigQuery
def load_ratings_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT user_id, title, rating
        FROM `{PROJECT_ID}.{DATASET_ID}.ratings_with_titles`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading ratings from BigQuery: {e}")
        raise

In [80]:
movies_bq = load_movies_bq()
movies_dict = {key: list(value) for key, value in movies_bq[['title', 'genres']].to_dict(orient='list').items()}

In [81]:
ratings_bq = load_ratings_bq()
ratings_dict = {key: list(value) for key, value in ratings_bq[['title', 'user_id', 'rating']].to_dict(orient='list').items()}

In [82]:
ratings = tf.data.Dataset.from_tensor_slices(ratings_dict)

In [83]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'rating': 2.0, 'title': b'Initiation', 'user_id': 56703}


In [84]:
movies = tf.data.Dataset.from_tensor_slices(movies_dict)

In [85]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]),
 'title': b'Siberian Sniper'}


In [86]:
ratings = ratings.map(lambda x: {
            "title": x["title"],
            "user_id": x["user_id"],
            "rating": x["rating"]
        })
movies = movies.map(lambda x: {
    "title": x["title"],
    "genres": x["genres"]
})

In [90]:
# print unique titles lenght from ratings and movies
print(f"Unique titles in ratings: {len(np.unique(ratings_dict['title']))}")
print(f"Unique titles in movies: {len(np.unique(movies_dict['title']))}")

Unique titles in ratings: 5283
Unique titles in movies: 5303


In [91]:
# Create a dictionary from the movies dataset
movies_dict = {movie["title"].numpy().decode('utf-8'): movie["genres"].numpy() for movie in movies}

# Combine the ratings and movies datasets only for the movies that are in the ratings dataset
def combine_datasets(rating):
    def lookup_genres(title):
        title_str = title.numpy().decode('utf-8')  # Convert to numpy and decode the title from bytes to string
        return movies_dict.get(title_str, [0] * 19)

    genres = tf.py_function(
        func=lookup_genres,
        inp=[rating["title"]],
        Tout=tf.int32
    )
    genres.set_shape([19])
    rating["genres"] = genres
    return rating

combined_dataset = ratings.map(combine_datasets)

In [92]:
# print one example of combined dataset
for x in combined_dataset.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
      dtype=int32),
 'rating': 2.0,
 'title': b'Initiation',
 'user_id': 56703}


In [98]:
combined_dataset = combined_dataset.map(lambda x: {
    "title": x["title"],
    "user_id": x["user_id"],
    "genres": x["genres"],
    "rating": x["rating"]
}, num_parallel_calls=tf.data.AUTOTUNE)

In [100]:
# Set the seed for reproducibility
tf.random.set_seed(42)

# Shuffle the dataset with a large buffer size
# Ensure the buffer size is large enough to cover randomness but not too large to exhaust memory
shuffle_buffer_size = 200000  # Smaller buffer size for faster shuffling
shuffled = combined_dataset.shuffle(buffer_size=shuffle_buffer_size, seed=42, reshuffle_each_iteration=True)

# Calculate relative proportions for splitting
train_ratio = 0.8

ds_length = int(tf.data.experimental.cardinality(shuffled).numpy())
print(f"Length of the dataset: {ds_length}")

# Define the split function for large datasets
def split_dataset(dataset, train_ratio):
    # Determine split points
    trainds = dataset.take(int(train_ratio * ds_length))
    testds = dataset.skip(int(train_ratio * ds_length))

    return trainds, testds

# Perform the split
trainds, testds = split_dataset(shuffled, train_ratio)

# Optimize performance with prefetching
train_batch_size =  128
eval_test_batch_size = 64

# Optimize datasets with batching, caching, and prefetching
trainds = trainds.batch(train_batch_size).cache().prefetch(tf.data.AUTOTUNE)
testds = testds.batch(eval_test_batch_size).cache().prefetch(tf.data.AUTOTUNE)

Length of the dataset: 174490


In [101]:
titles = combined_dataset.batch(100000).map(lambda x: x["title"])
user_ids = combined_dataset.batch(1000000).map(lambda x: x["user_id"])
genres = combined_dataset.batch(100000).map(lambda x: {
        "title": x["title"],
        "genres": x["genres"]
    })

In [102]:
unique_titles = np.unique(np.concatenate(list(titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))
unique_genres = [
            'Action',
            'Adventure',
            'Animation',
            'Children',
            'Comedy',
            'Crime',
            'Drama',
            'Documentary',
            'Fantasy',
            'Film-Noir',
            'Horror',
            'IMAX',
            'Musical',
            'Mystery',
            'Romance',
            'Sci-Fi',
            'Thriller',
            'War',
            'Western'
        ]

In [103]:
print(f"Unique titles: {len(unique_titles)}")

Unique titles: 5283


In [104]:
class RecommendationModel(tfrs.Model):
    def __init__(self, user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight=1.0, retrieval_weight=1.0):
        super().__init__()
        self.user_model = user_model
        self.movie_model = movie_model
        self.genre_model = genre_model
        self.rating_model = rating_model
        self.rating_task = rating_task
        self.retrieval_task = retrieval_task
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def call(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        movie_embeddings = self.movie_model(features["title"])
        genre_embeddings = self.genre_model(features["genres"])
        rating_predictions = self.rating_model([features["user_id"], features["title"], features["genres"]])
        return user_embeddings, movie_embeddings, genre_embeddings, rating_predictions

    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        ratings = features.pop("rating")
        user_embeddings, movie_embeddings, genre_embeddings, rating_predictions = self(features)
        rating_loss = self.rating_task(labels=ratings, predictions=rating_predictions)
        retrieval_loss = self.retrieval_task(user_embeddings, movie_embeddings)
        return (self.rating_weight * rating_loss
                + self.retrieval_weight * retrieval_loss)

In [18]:
# # Function to create the model
# def create_model(unique_user_ids, unique_movie_ids, num_genres, rating_weight=1.0, retrieval_weight=1.0):
#     embedding_dimension = 128
#     user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
#     movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
#     genre_input = tf.keras.layers.Input(shape=(num_genres,), dtype=tf.float32, name='genres')

#     user_lookup = tf.keras.layers.IntegerLookup(vocabulary=unique_user_ids, mask_token=None)
#     movie_lookup = tf.keras.layers.StringLookup(vocabulary=unique_titles, mask_token=None)

#     user_embedding = tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
#     movie_embedding = tf.keras.layers.Embedding(len(unique_movie_ids) + 1, embedding_dimension)(movie_lookup(movie_input))
#     genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

#     concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

#     dense_1 = tf.keras.layers.Dense(256, activation="relu")(concatenated_embeddings)
#     dropout_1 = tf.keras.layers.Dropout(0.5)(dense_1)
#     dense_2 = tf.keras.layers.Dense(128, activation="relu")(dropout_1)
#     dropout_2 = tf.keras.layers.Dropout(0.5)(dense_2)
#     rating_output = tf.keras.layers.Dense(1)(dropout_2)


#     user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
#     movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
#     genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
#     rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

#     metrics = tfrs.metrics.FactorizedTopK(
#         candidates=tf.data.Dataset.from_tensor_slices(unique_movie_ids).batch(128).map(movie_model)
#     )
#     rating_task = tfrs.tasks.Ranking(
#         loss=tf.keras.losses.MeanSquaredError(),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()],
#     )
#     retrieval_task = tfrs.tasks.Retrieval(
#         metrics=metrics
#     )

#     model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight, retrieval_weight)
#     lr_schedule_ed = tf.keras.optimizers.schedules.ExponentialDecay(
#         initial_learning_rate=0.1,
#         decay_steps=545,
#         decay_rate=0.9,
#         staircase=True
#     )

#     # lr_schedule_pcd = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
#     #     boundaries=[2200, 3000, 3500],
#     #     values=[0.1, 0.01, 0.001, 0.0001]
#     # )
    
#     # Use the learning rate schedule in the optimizer
#     model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=lr_schedule_ed))
#     return model


In [105]:
class RecommendationHyperModel(HyperModel):
    def __init__(self, unique_user_ids, unique_titles, num_genres, rating_weight=1.0, retrieval_weight=1.0):
        self.unique_user_ids = unique_user_ids
        self.unique_titles = unique_titles
        self.num_genres = num_genres
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def build(self, hp):
        embedding_dimension = hp.Int('embedding_dimension', min_value=32, max_value=256, step=32)
        user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
        movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
        genre_input = tf.keras.layers.Input(shape=(self.num_genres,), dtype=tf.float32, name='genres')

        user_lookup = tf.keras.layers.IntegerLookup(vocabulary=self.unique_user_ids, mask_token=None)
        movie_lookup = tf.keras.layers.StringLookup(vocabulary=self.unique_titles, mask_token=None)

        user_embedding = tf.keras.layers.Embedding(len(self.unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
        movie_embedding = tf.keras.layers.Embedding(len(self.unique_titles) + 1, embedding_dimension)(movie_lookup(movie_input))
        genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

        concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

        dense_1 = tf.keras.layers.Dense(hp.Int('units_1', min_value=128, max_value=512, step=64), activation="relu")(concatenated_embeddings)
        dropout_1 = tf.keras.layers.Dropout(hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1))(dense_1)
        dense_2 = tf.keras.layers.Dense(hp.Int('units_2', min_value=64, max_value=256, step=32), activation="relu")(dropout_1)
        dropout_2 = tf.keras.layers.Dropout(hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1))(dense_2)
        rating_output = tf.keras.layers.Dense(1)(dropout_2)

        user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
        movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
        genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
        rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

        metrics = tfrs.metrics.FactorizedTopK(
            candidates=tf.data.Dataset.from_tensor_slices(self.unique_titles).batch(128).map(movie_model)
        )
        rating_task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()],
        )
        retrieval_task = tfrs.tasks.Retrieval(
            metrics=metrics
        )
        model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, self.rating_weight, self.retrieval_weight)
        
        # Define the learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log'),
            decay_steps=hp.Int('decay_steps', min_value=500, max_value=1000, step=100),
            decay_rate=hp.Float('decay_rate', min_value=0.8, max_value=0.9, step=0.05),
            staircase=True
        )

        # # Define the learning rate schedule using CosineDecay
        # lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        #     initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-1, sampling='log'),
        #     decay_steps=1000,
        #     alpha=0.1
        # )
        
        # Use the learning rate schedule in the optimizer
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule))
        return model

In [179]:
tuner = Hyperband(
    RecommendationHyperModel(unique_user_ids, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_5_categorical_accuracy", direction="max"),
    max_epochs=12,
    # max_trials=10,
    factor=3,
    # executions_per_trial=1,
    directory='tuning/tpe',
    project_name='20250122063906/movie_recommendation',
)

# Reload the tuner to load previous results
# tuner.reload()

tuner.search(trainds, epochs=12, validation_data=testds, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)])

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
tuned_model = tuner.hypermodel.build(best_hps)

Reloading Tuner from tuning/tpe/20250122063906/movie_recommendation/tuner0.json


In [180]:
best_hps.values

{'embedding_dimension': 224,
 'units_1': 320,
 'dropout_1': 0.4,
 'units_2': 224,
 'dropout_2': 0.4,
 'learning_rate': 0.0022820322663096213,
 'decay_steps': 1000,
 'decay_rate': 0.8500000000000001,
 'tuner/epochs': 12,
 'tuner/initial_epoch': 4,
 'tuner/bracket': 2,
 'tuner/round': 2,
 'tuner/trial_id': '0013'}

In [109]:
def generate_unique_genres(n):
    """
    Generate a one-hot encoded array for `n` unique genres.
    
    Args:
        n (int): Number of unique genres.
    
    Returns:
        np.ndarray: A one-hot encoded array of shape (n, n).
    """
    return np.eye(n, dtype=int).tolist()
    
unique_genres_array = generate_unique_genres(len(unique_genres))

# Extract user and movie embeddings
user_embeddings = tuned_model.user_model.predict(unique_user_ids)
movie_embeddings = tuned_model.movie_model.predict(unique_titles)
genre_embeddings = tuned_model.genre_model.predict(unique_genres_array)


# Create a dictionary to map user IDs and movie IDs to their embeddings
user_embedding_dict = {user_id: embedding for user_id, embedding in zip(unique_user_ids, user_embeddings)}
movie_embedding_dict = {movie_id: embedding for movie_id, embedding in zip(unique_titles, movie_embeddings)}
genre_embedding_dict = {i: embedding for i, embedding in enumerate(genre_embeddings)}

# user_embedding_dict = {k: v for k, v in user_embedding_dict.items()}
movie_embedding_dict = {k.decode('utf-8'): v for k, v in movie_embedding_dict.items()}
# genre_embedding_dict = {k: v for k, v in genre_embedding_dict.items()}

1/1 [==============================] - 0s 113ms/step


In [110]:
# Convert combined dataset to pandas DataFrame for xgb
xgb_df = pd.DataFrame(combined_dataset.as_numpy_iterator())
print(xgb_df.head())

                          title  user_id  \
0                 b'Initiation'    56703   
1              b'Anything Goes'   125172   
2  b'1000 Miles From Christmas'   129858   
3                   b'Eternals'   174766   
4                   b'Eternals'    55681   

                                              genres  rating  
0  [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...     2.0  
1  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     2.0  
2  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     2.0  
3  [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     2.0  
4  [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     2.0  


In [ ]:
# Create feature vectors by combining user, movie, and genres embeddings
def create_feature_vector(row):
    user_id = row['user_id']
    title = row['title'].decode('utf-8')
    genres = row['genres']
    
    if user_id not in user_embedding_dict:
        raise KeyError(f"User ID {user_id} not found in user_embedding_dict")
    if title not in movie_embedding_dict:
        raise KeyError(f"Title '{title}' not found in movie_embedding_dict")
    
    user_embedding = user_embedding_dict[user_id]
    movie_embedding = movie_embedding_dict[title]
    
    # Calculate the genre embedding by multiplying the one-hot vector with the genre embeddings
    # genre_embedding = np.dot(genres, np.array([genre_embedding_dict[i] for i in range(len(genres))]))
    
    return np.concatenate([user_embedding, movie_embedding, genres])

# Apply the function to create feature vectors
try:
    xgb_df['features'] = xgb_df.apply(create_feature_vector, axis=1)
except KeyError as e:
    print(e)

# Set the seed for reproducibility
random_seed = 42

# Shuffle the DataFrame
shuffled_df = xgb_df.sample(frac=1, random_state=random_seed).reset_index(drop=True)
train_df, val_df = train_test_split(shuffled_df, test_size=0.2, random_state=random_seed)

# Identify overlapping rows
common_rows = train_df[['user_id', 'title']].merge(val_df[['user_id', 'title']], how='inner')

# Remove overlapping rows from the validation set
if not common_rows.empty:
    val_df = val_df[~val_df[['user_id', 'title']].apply(tuple, axis=1).isin(common_rows.apply(tuple, axis=1))]
    print(str(len(common_rows)) + " overlapping rows removed from the validation set.")

assert train_df[['user_id', 'title']].merge(val_df[['user_id', 'title']], how='inner').empty, "Data leakage detected!"

X_train = np.vstack(train_df['features'].values)
y_train = train_df['rating'].values
X_val = np.vstack(val_df['features'].values)
y_val = val_df['rating'].values

# Add content-based features to the feature matrix
additional_train_features = train_df.drop(columns=['user_id', 'title', 'rating', 'genres', 'features']).values
additional_val_features = val_df.drop(columns=['user_id', 'title', 'rating', 'genres', 'features']).values

# Concatenate the additional features
X_train = np.hstack([X_train, additional_train_features])
X_val = np.hstack([X_val, additional_val_features])

# Verify the shapes of the feature matrices
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")

# Group sizes for ranking
group_train = train_df.groupby('user_id').size().to_list()
group_val = val_df.groupby('user_id').size().to_list()

# Verify that the sum of group sizes matches the number of rows
assert sum(group_train) == X_train.shape[0], "Mismatch between group sizes and number of rows in X_train"
assert sum(group_val) == X_val.shape[0], "Mismatch between group sizes and number of rows in X_val"

# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(group_train)
dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(group_val)

10 overlapping rows removed from the validation set.
Shape of X_train: (139592, 467)
Shape of X_val: (34888, 467)


In [161]:
print(xgb_df[['title', 'features']])

                               title  \
0                      b'Initiation'   
1                   b'Anything Goes'   
2       b'1000 Miles From Christmas'   
3                        b'Eternals'   
4                        b'Eternals'   
...                              ...   
174485     b'The Exorcist: Believer'   
174486                     b'Caviar'   
174487             b'The Blackening'   
174488             b'The Blackening'   
174489                b'Alibi.com 2'   

                                                 features  
0       [-0.019695067778229713, -0.015210390090942383,...  
1       [0.028149042278528214, 0.0034996643662452698, ...  
2       [0.023759040981531143, -0.04796649143099785, 0...  
3       [0.005890071392059326, 0.003522120416164398, -...  
4       [0.03855634853243828, 0.03402772173285484, 0.0...  
...                                                   ...  
174485  [0.007346700876951218, -0.03439323976635933, -...  
174486  [-0.0045740604400634766, -0.035

In [162]:
print(train_df.head())

                                         title  user_id  \
110516                         b'Looop Lapeta'    55707   
149939                            b'Danny Boy'   197432   
96558                   b'Father of the Bride'   156823   
48034            b'The Greatest Beer Run Ever'   173111   
32060   b'Spider-Man: Across the Spider-Verse'   167721   

                                                   genres  rating  \
110516  [1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...     1.5   
149939  [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     3.0   
96558   [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     0.5   
48034   [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     5.0   
32060   [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...     5.0   

                                                 features  
110516  [0.02188045158982277, 0.038019727915525436, 0....  
149939  [0.007408477365970612, -0.002667464315891266, ...  
96558   [-0.002238534390926361, -0.003744818270206

In [163]:
print(val_df.head())

                                          title  user_id  \
70066  b"Are You There God? It's Me, Margaret."   163298   
38489                                   b'Nope'   178415   
7191                                    b'Dune'   105780   
37294                              b'Amsterdam'   142467   
88685                b'Ghostbusters: Afterlife'    68997   

                                                  genres  rating  \
70066  [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     4.0   
38489  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...     2.5   
7191   [1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...     3.5   
37294  [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, ...     2.0   
88685  [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     3.5   

                                                features  
70066  [0.04523665830492973, 0.00799490138888359, -0....  
38489  [0.03897855058312416, -0.031281985342502594, -...  
7191   [0.033266257494688034, 0.036346565932035446, -

In [164]:
# Define the objective function for Optuna
def objective(trial):
    param = {
        'objective': 'rank:pairwise',
        'eval_metric': 'ndcg',
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'lambda': trial.suggest_float('lambda', 0.0, 1.0),
    }
    
    bst = xgb.train(param, dtrain, num_boost_round=100, evals=[(dval, 'eval')], early_stopping_rounds=10, verbose_eval=True)
    y_pred = bst.predict(dval)
    min_rating = 0.5
    max_rating = 5.0
    min_pred = np.min(y_pred)
    max_pred = np.max(y_pred)
    y_pred_scaled = min_rating + (y_pred - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)
    
    # Calculate NDCG score
    ndcg = ndcg_score([y_val], [y_pred_scaled])
    return ndcg

# Create a study and optimize the objective function
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Get the best parameters
best_params = study.best_params
print(f"Best parameters: {best_params}")

# Train the model with the best parameters
bst = xgb.train(best_params, dtrain, num_boost_round=100, evals=[(dval, 'eval')], early_stopping_rounds=10)

[I 2025-01-24 22:49:18,743] A new study created in memory with name: no-name-54dbe43f-1f29-4ac5-bb11-4823048fa6ee


[0]	eval-ndcg:0.94089
[1]	eval-ndcg:0.94207
[2]	eval-ndcg:0.94242
[3]	eval-ndcg:0.94253
[4]	eval-ndcg:0.94296
[5]	eval-ndcg:0.94318
[6]	eval-ndcg:0.94345
[7]	eval-ndcg:0.94355
[8]	eval-ndcg:0.94351
[9]	eval-ndcg:0.94366
[10]	eval-ndcg:0.94374
[11]	eval-ndcg:0.94385
[12]	eval-ndcg:0.94414
[13]	eval-ndcg:0.94402
[14]	eval-ndcg:0.94365
[15]	eval-ndcg:0.94391
[16]	eval-ndcg:0.94410
[17]	eval-ndcg:0.94424
[18]	eval-ndcg:0.94425
[19]	eval-ndcg:0.94433
[20]	eval-ndcg:0.94425
[21]	eval-ndcg:0.94443
[22]	eval-ndcg:0.94455
[23]	eval-ndcg:0.94449
[24]	eval-ndcg:0.94455
[25]	eval-ndcg:0.94459
[26]	eval-ndcg:0.94467
[27]	eval-ndcg:0.94461
[28]	eval-ndcg:0.94468
[29]	eval-ndcg:0.94485
[30]	eval-ndcg:0.94495
[31]	eval-ndcg:0.94517
[32]	eval-ndcg:0.94524
[33]	eval-ndcg:0.94525
[34]	eval-ndcg:0.94508
[35]	eval-ndcg:0.94548
[36]	eval-ndcg:0.94593
[37]	eval-ndcg:0.94645
[38]	eval-ndcg:0.94631
[39]	eval-ndcg:0.94668
[40]	eval-ndcg:0.94683
[41]	eval-ndcg:0.94699
[42]	eval-ndcg:0.94703
[43]	eval-ndcg:0.9472

[I 2025-01-24 22:50:39,823] Trial 0 finished with value: 0.9851095560810184 and parameters: {'eta': 0.0840648663231847, 'max_depth': 6, 'min_child_weight': 4, 'subsample': 0.8575220828560401, 'colsample_bytree': 0.7406083358197846, 'gamma': 0.24016266863643043, 'lambda': 0.6345056977997102}. Best is trial 0 with value: 0.9851095560810184.


[0]	eval-ndcg:0.93867
[1]	eval-ndcg:0.93924
[2]	eval-ndcg:0.93949
[3]	eval-ndcg:0.93998
[4]	eval-ndcg:0.94050
[5]	eval-ndcg:0.94113
[6]	eval-ndcg:0.94142
[7]	eval-ndcg:0.94162
[8]	eval-ndcg:0.94201
[9]	eval-ndcg:0.94179
[10]	eval-ndcg:0.94219
[11]	eval-ndcg:0.94222
[12]	eval-ndcg:0.94218
[13]	eval-ndcg:0.94230
[14]	eval-ndcg:0.94282
[15]	eval-ndcg:0.94279
[16]	eval-ndcg:0.94299
[17]	eval-ndcg:0.94303
[18]	eval-ndcg:0.94319
[19]	eval-ndcg:0.94322
[20]	eval-ndcg:0.94363
[21]	eval-ndcg:0.94362
[22]	eval-ndcg:0.94386
[23]	eval-ndcg:0.94371
[24]	eval-ndcg:0.94376
[25]	eval-ndcg:0.94383
[26]	eval-ndcg:0.94429
[27]	eval-ndcg:0.94414
[28]	eval-ndcg:0.94418
[29]	eval-ndcg:0.94449
[30]	eval-ndcg:0.94465
[31]	eval-ndcg:0.94482
[32]	eval-ndcg:0.94501
[33]	eval-ndcg:0.94523
[34]	eval-ndcg:0.94513
[35]	eval-ndcg:0.94534
[36]	eval-ndcg:0.94550
[37]	eval-ndcg:0.94589
[38]	eval-ndcg:0.94605
[39]	eval-ndcg:0.94602
[40]	eval-ndcg:0.94626
[41]	eval-ndcg:0.94615
[42]	eval-ndcg:0.94612
[43]	eval-ndcg:0.9462

[I 2025-01-24 22:51:33,576] Trial 1 finished with value: 0.984337682794146 and parameters: {'eta': 0.16065190310287736, 'max_depth': 4, 'min_child_weight': 3, 'subsample': 0.6956626395414864, 'colsample_bytree': 0.7899705137013745, 'gamma': 0.04122518539853093, 'lambda': 0.32352171911360494}. Best is trial 0 with value: 0.9851095560810184.


...

[0]	eval-ndcg:0.94039
[1]	eval-ndcg:0.94130
[2]	eval-ndcg:0.94213
[3]	eval-ndcg:0.94263
[4]	eval-ndcg:0.94290
[5]	eval-ndcg:0.94329
[6]	eval-ndcg:0.94329
[7]	eval-ndcg:0.94357
[8]	eval-ndcg:0.94387
[9]	eval-ndcg:0.94394
[10]	eval-ndcg:0.94416
[11]	eval-ndcg:0.94421
[12]	eval-ndcg:0.94398
[13]	eval-ndcg:0.94395
[14]	eval-ndcg:0.94462
[15]	eval-ndcg:0.94517
[16]	eval-ndcg:0.94516
[17]	eval-ndcg:0.94551
[18]	eval-ndcg:0.94599
[19]	eval-ndcg:0.94614
[20]	eval-ndcg:0.94620
[21]	eval-ndcg:0.94657
[22]	eval-ndcg:0.94702
[23]	eval-ndcg:0.94762
[24]	eval-ndcg:0.94770
[25]	eval-ndcg:0.94819
[26]	eval-ndcg:0.94833
[27]	eval-ndcg:0.94887
[28]	eval-ndcg:0.94907
[29]	eval-ndcg:0.94915
[30]	eval-ndcg:0.94954
[31]	eval-ndcg:0.94955
[32]	eval-ndcg:0.94953
[33]	eval-ndcg:0.94977
[34]	eval-ndcg:0.94986
[35]	eval-ndcg:0.94987
[36]	eval-ndcg:0.95006
[37]	eval-ndcg:0.95042
[38]	eval-ndcg:0.95049
[39]	eval-ndcg:0.95082
[40]	eval-ndcg:0.95073
[41]	eval-ndcg:0.95073
[42]	eval-ndcg:0.95073
[43]	eval-ndcg:0.9509

[I 2025-01-25 00:10:28,109] Trial 48 finished with value: 0.9862778404724598 and parameters: {'eta': 0.22249231192239866, 'max_depth': 5, 'min_child_weight': 2, 'subsample': 0.9656716472301027, 'colsample_bytree': 0.6992084885749975, 'gamma': 0.3245329500843061, 'lambda': 0.4166542194063301}. Best is trial 39 with value: 0.9876102731943961.


[0]	eval-ndcg:0.93820
[1]	eval-ndcg:0.94159
[2]	eval-ndcg:0.94297
[3]	eval-ndcg:0.94369
[4]	eval-ndcg:0.94446
[5]	eval-ndcg:0.94453
[6]	eval-ndcg:0.94447
[7]	eval-ndcg:0.94551
[8]	eval-ndcg:0.94552
[9]	eval-ndcg:0.94579
[10]	eval-ndcg:0.94608
[11]	eval-ndcg:0.94629
[12]	eval-ndcg:0.94672
[13]	eval-ndcg:0.94669
[14]	eval-ndcg:0.94687
[15]	eval-ndcg:0.94727
[16]	eval-ndcg:0.94891
[17]	eval-ndcg:0.94914
[18]	eval-ndcg:0.94986
[19]	eval-ndcg:0.95007
[20]	eval-ndcg:0.95049
[21]	eval-ndcg:0.95091
[22]	eval-ndcg:0.95107
[23]	eval-ndcg:0.95141
[24]	eval-ndcg:0.95155
[25]	eval-ndcg:0.95171
[26]	eval-ndcg:0.95217
[27]	eval-ndcg:0.95251
[28]	eval-ndcg:0.95324
[29]	eval-ndcg:0.95371
[30]	eval-ndcg:0.95379
[31]	eval-ndcg:0.95388
[32]	eval-ndcg:0.95422
[33]	eval-ndcg:0.95424
[34]	eval-ndcg:0.95443
[35]	eval-ndcg:0.95464
[36]	eval-ndcg:0.95509
[37]	eval-ndcg:0.95508
[38]	eval-ndcg:0.95509
[39]	eval-ndcg:0.95570
[40]	eval-ndcg:0.95581
[41]	eval-ndcg:0.95600
[42]	eval-ndcg:0.95597
[43]	eval-ndcg:0.9562

[I 2025-01-25 00:12:55,325] Trial 49 finished with value: 0.9875062315325805 and parameters: {'eta': 0.20513977585114931, 'max_depth': 10, 'min_child_weight': 8, 'subsample': 0.895743602835286, 'colsample_bytree': 0.8908377688361898, 'gamma': 0.15923164834190737, 'lambda': 0.7738409688841434}. Best is trial 39 with value: 0.9876102731943961.


Best parameters: {'eta': 0.18015987625848606, 'max_depth': 10, 'min_child_weight': 3, 'subsample': 0.964521898660231, 'colsample_bytree': 0.7226329024799188, 'gamma': 0.19998509587001428, 'lambda': 0.26987835590612563}
[0]	eval-rmse:2.54963
[1]	eval-rmse:2.16412
[2]	eval-rmse:1.86034
[3]	eval-rmse:1.62375
[4]	eval-rmse:1.44225
[5]	eval-rmse:1.30438
[6]	eval-rmse:1.20273
[7]	eval-rmse:1.11960
[8]	eval-rmse:1.06523
[9]	eval-rmse:1.02342
[10]	eval-rmse:0.99416
[11]	eval-rmse:0.96576
[12]	eval-rmse:0.94708
[13]	eval-rmse:0.93296
[14]	eval-rmse:0.92540
[15]	eval-rmse:0.91853
[16]	eval-rmse:0.91180
[17]	eval-rmse:0.90833
[18]	eval-rmse:0.90190
[19]	eval-rmse:0.89684
[20]	eval-rmse:0.88991
[21]	eval-rmse:0.88838
[22]	eval-rmse:0.88640
[23]	eval-rmse:0.88444
[24]	eval-rmse:0.88151
[25]	eval-rmse:0.87902
[26]	eval-rmse:0.87662
[27]	eval-rmse:0.87494
[28]	eval-rmse:0.87418
[29]	eval-rmse:0.87308
[30]	eval-rmse:0.87219
[31]	eval-rmse:0.87121
[32]	eval-rmse:0.86823
[33]	eval-rmse:0.86786
[34]	eval

In [140]:
# print lenght unique val df titles
print(len(np.unique(list(val_df['title']))))

2801


In [168]:
def rank_movies(user_id, combined_dataset, bst, k=5):
    if user_id not in user_embedding_dict:
        raise ValueError(f"User ID {user_id} not found in user_embedding_dict")

    # Filter the dataset for the given user
    user_data = combined_dataset[combined_dataset['user_id'] == user_id]
    candidate_features = np.vstack(user_data['features'].values)
    
    dtest = xgb.DMatrix(candidate_features)
    predicted_ratings = bst.predict(dtest)

    # Scale ratings (optional)
    min_rating, max_rating = 0.5, 5.0
    min_pred = np.min(predicted_ratings)
    max_pred = np.max(predicted_ratings)
    predicted_ratings_scaled = min_rating + (predicted_ratings - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)
    
    ranked_movies = sorted(zip(predicted_ratings_scaled, user_data['title'].values), reverse=True)
    return ranked_movies[:k]


In [238]:
# Predict ratings for the validation set
y_pred = bst.predict(dval)
min_rating = 0.5
max_rating = 5.0
min_pred = np.min(y_pred)
max_pred = np.max(y_pred)
y_pred_scaled = min_rating + (y_pred - min_pred) * (max_rating - min_rating) / (max_pred - min_pred)

# Assuming val_df contains the validation data with columns: user_id, title, rating
val_df['predicted_rating'] = y_pred_scaled

# Rank items for each user
val_df['rank'] = val_df.groupby('user_id')['predicted_rating'].rank(ascending=False, method='first')
print(pd.Series(y_pred_scaled).describe())

unique_titles_nb = [title.encode('utf-8') for title in movie_embedding_dict.keys()]

# Example usage
user_id = 163298


top_k_movies = rank_movies(user_id, xgb_df, bst, k=5)
print(f'Ranked movies for user {user_id}: {top_k_movies}')

count    34888.000000
mean         3.319079
std          0.482458
min          0.500000
25%          3.053221
50%          3.370281
75%          3.647004
max          5.000000
dtype: float64
Ranked movies for user 163298: [(5.0, b'Everything Everywhere All at Once'), (4.9339776, b'Dune'), (4.6281533, b'Top Gun: Maverick'), (4.570015, b'The Banshees of Inisherin'), (4.5477934, b'The Rescue')]


In [ ]:
# Calculate NDCG score
ndcg = ndcg_score([y_val], [y_pred_scaled])
print(f'NDCG Score: {ndcg}')

# Determine relevant items with a higher threshold (e.g., rating > 3.5)
def get_relevant_items(user_data, threshold=3.5):
    if user_data['rating'].max() > threshold:
        return user_data[user_data['rating'] > threshold]['title'].values
    else:
        return user_data['title'].values

# Calculate Precision@K and Recall@K and Accuracy@K
def precision_at_k(val_df, k, threshold=3.5):
    precision_scores = []
    for user_id in val_df['user_id'].unique():
        user_data = val_df[val_df['user_id'] == user_id]
        true_titles = get_relevant_items(user_data, threshold)  # Determine relevant items with the new threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        
        # Calculate precision
        precision = len(set(true_titles).intersection(top_k_titles)) / k
        precision_scores.append(precision)
    return np.mean(precision_scores)

def recall_at_k(val_df, k, threshold=3.5):
    recall_scores = []
    for user_id in val_df['user_id'].unique():
        user_data = val_df[val_df['user_id'] == user_id]
        true_titles = get_relevant_items(user_data, threshold)  # Determine relevant items with the new threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        
        # Calculate recall
        recall = len(set(true_titles).intersection(top_k_titles)) / len(true_titles)
        recall_scores.append(recall)
    return np.mean(recall_scores)

# Calculate Accuracy@K
def accuracy_at_k(val_df, k, threshold=3.5):
    correct_count = 0
    total_users = val_df['user_id'].nunique()
    
    for user_id in val_df['user_id'].unique():
        user_data = val_df[val_df['user_id'] == user_id]
        true_titles = get_relevant_items(user_data, threshold)  # Determine relevant items with the new threshold
        top_k_titles = user_data.nsmallest(k, 'rank')['title'].values
        
        # Check if any true title is in the top-K predictions
        if set(true_titles).intersection(top_k_titles):
            correct_count += 1
    
    return correct_count / total_users

# Evaluate the impact of changing the threshold
threshold = 2.5
precision_10 = precision_at_k(val_df, 1, threshold)
recall_10 = recall_at_k(val_df, 1, threshold)
accuracy_10 = accuracy_at_k(val_df, 1, threshold)

print(f'Precision@10 with threshold {threshold}: {precision_10}')
print(f'Recall@10 with threshold {threshold}: {recall_10}')
print(f'Accuracy@10 with threshold {threshold}: {accuracy_10}')

NDCG Score: 0.9882507480857663
Precision@10 with threshold 2.5: 0.28151715833835045
Recall@10 with threshold 2.5: 0.9716499060630412
Accuracy@10 with threshold 2.5: 0.9998795906080674


In [ ]:

# Additional diagnostics
print("Distribution of Ratings in Training Set:")
print(train_df['rating'].value_counts())

print("Distribution of Ratings in Validation Set:")
print(val_df['rating'].value_counts())

print("Predictions Distribution:")
print(pd.Series(y_pred_scaled).describe())

In [248]:
# print genre for given movieId using movies_bq
def get_genre(title):
    try:
        return movies_bq[movies_bq['title'] == title]['genres'].values[0]
    except Exception as e:
        print(e)

def rate_movies_with_hypermodel(hypermodel, user_id, titles, genres):
    # Use the recommendation hypermodel to predict ratings for the given user and movie titles
    predicted_ratings = []
    try:
        for title, genre in zip(titles, genres):
            # Predict ratings using the hypermodel
            _, _, _, predicted_rating = hypermodel({
                "user_id": np.array([user_id]),
                "title": np.array([title]),
                "genres": np.array([genre])
            })
            predicted_ratings.append([title, predicted_rating.numpy()[0][0]])
        return predicted_ratings
    except Exception as e:
        print(e)

movie_titles = ['The Rescue', 'Siberian Sniper']
movie_genres = [get_genre(title) for title in movie_titles]
predicted_ratings = rate_movies_with_hypermodel(tuned_model, user_id, movie_titles, movie_genres)
print("Predicted ratings:")
print(predicted_ratings)

Predicted ratings:
[['The Rescue', 3.073621], ['Siberian Sniper', 2.293641]]


In [258]:
def get_final_predictions(user_id, combined_dataset, bst, hypermodel, k=5):
    # Rank movies using the XGBoost model
    top_k_movies = rank_movies(user_id, combined_dataset, bst, k*5)

    # Get the movie titles and genres
    _, movie_titles = zip(*top_k_movies)

    # Decode the movie titles
    movie_titles = [title.decode('utf-8') for title in movie_titles]
    print(movie_titles)
    movie_genres = [get_genre(title) for title in movie_titles]
    print(movie_genres)

    # Predict ratings using the hypermodel
    final_predictions = rate_movies_with_hypermodel(hypermodel, user_id, movie_titles, movie_genres)

    # Sort the final predictions by rating
    final_predictions = sorted(final_predictions, key=lambda x: x[1], reverse=True)

    return final_predictions[:k]

In [259]:
final_predictions = get_final_predictions(user_id, xgb_df, bst, tuned_model, k=5)
print("Final Predictions:")
print(final_predictions)

['Everything Everywhere All at Once', 'Dune', 'Top Gun: Maverick', 'The Banshees of Inisherin', 'The Rescue', 'The Worst Person in the World', 'Petite maman', 'Aftersun', 'Spider-Man: No Way Home', 'Nobody', 'The Batman', 'CODA', 'Pearl', 'TÁR', 'Triangle of Sadness', 'The Fabelmans', 'The Alpinist', 'Living', 'Decision to Leave', 'The Souvenir Part II', 'TINA', 'The Last Duel', 'Untold: Malice at the Palace', 'Navalny', 'All Light, Everywhere']
[array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]), array([1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]), array([1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), array([0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]), array([0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), array([0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), array([1, 1, 0, 0, 0,

In [48]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/tpe/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_5_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(f"trained_model/tpe/tensorboard/{timestamp}_cp.ckpt", histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

In [31]:
model = create_model(unique_user_ids, unique_titles, len(unique_genres), rating_weight=1.0, retrieval_weight=1.0)

In [ ]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/bayesian/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_5_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(f"trained_model/bayesian/tensorboard/{timestamp}_cp.ckpt", histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

bayesian_tuner = BayesianOptimization(
    RecommendationHyperModel(unique_user_ids, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_5_categorical_accuracy", direction="max"),
    max_trials=10,
    executions_per_trial=1,
    directory='tuning/bayesian_optimization',
    project_name='20250120133706/movie_recommendation',
)
bayesian_tuner.reload()
# Get the best hyperparameters
best_hps = bayesian_tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
bayesian_model = bayesian_tuner.hypermodel.build(best_hps)
# Build the model by calling it on a batch of data
bayesian_model.fit(
    trainds,
    epochs=10,
    # validation_data=testds,
    callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

In [ ]:
# Build the model by calling it on a batch of data
model.build(input_shape={
	"user_id": (None,),
	"title": (None,),
	"genres": (None, len(unique_genres))
})

In [181]:
# Train the model
history = tuned_model.fit(
    trainds,
    epochs=12,
    # steps_per_epoch=5,
    # validation_data=testds,
    # callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

Epoch 1/12


2025-01-25 12:56:29.896650: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 43366 of 200000
2025-01-25 12:56:39.896992: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 92813 of 200000
2025-01-25 12:56:49.896660: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 144486 of 200000


   1/1091 [..............................] - ETA: 12:27:33 - root_mean_squared_error: 3.5189 - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_100_categorical_accuracy: 0.0156 - loss: 633.6000 - regularization_loss: 0.0000e+00 - total_loss: 633.6000

2025-01-25 12:56:57.818769: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


1091/1091 [==============================] - 178s 126ms/step - root_mean_squared_error: 0.9338 - factorized_top_k/top_1_categorical_accuracy: 2.5789e-04 - factorized_top_k/top_5_categorical_accuracy: 0.0255 - factorized_top_k/top_10_categorical_accuracy: 0.0559 - factorized_top_k/top_50_categorical_accuracy: 0.1818 - factorized_top_k/top_100_categorical_accuracy: 0.2441 - loss: 604.9556 - regularization_loss: 0.0000e+00 - total_loss: 604.9556
Epoch 2/12
1091/1091 [==============================] - 138s 127ms/step - root_mean_squared_error: 0.8287 - factorized_top_k/top_1_categorical_accuracy: 0.0019 - factorized_top_k/top_5_categorical_accuracy: 0.1161 - factorized_top_k/top_10_categorical_accuracy: 0.1914 - factorized_top_k/top_50_categorical_accuracy: 0.4235 - factorized_top_k/top_100_categorical_accuracy: 0.5331 - loss: 529.6882 - regularization_loss: 0.0000e+00 - total_loss: 529.6882
Epoch 3/12
1091/1091 [==============================] - 152s 139ms/step - root_mean_squared_error: 

In [183]:
steps = ds_length * 0.2 // eval_test_batch_size
metrics = tuned_model.evaluate(testds, steps=steps, return_dict=True)

2025-01-25 13:29:13.436242: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 50764 of 200000
2025-01-25 13:29:23.436261: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 97148 of 200000
2025-01-25 13:29:33.436248: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 146981 of 200000
2025-01-25 13:29:38.519772: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


545/545 [==============================] - 80s 80ms/step - root_mean_squared_error: 0.6956 - factorized_top_k/top_1_categorical_accuracy: 0.0410 - factorized_top_k/top_5_categorical_accuracy: 0.2312 - factorized_top_k/top_10_categorical_accuracy: 0.3573 - factorized_top_k/top_50_categorical_accuracy: 0.6325 - factorized_top_k/top_100_categorical_accuracy: 0.7230 - loss: 163.8269 - regularization_loss: 0.0000e+00 - total_loss: 163.8269


In [ ]:
print(f"Retrieval top-1 accuracy: {metrics['factorized_top_k/top_1_categorical_accuracy']:.3f}.")
print(f"Retrieval top-5 accuracy: {metrics['factorized_top_k/top_5_categorical_accuracy']:.3f}.")
print(f"Retrieval top-10 accuracy: {metrics['factorized_top_k/top_10_categorical_accuracy']:.3f}.")
print(f"Ranking RMSE: {metrics['root_mean_squared_error']:.3f}.")

In [53]:
tuned_model.save_weights(f"trained_model/tpe/weights/{timestamp}_weights.h5")

In [ ]:


trained_user_embeddings, trained_movie_embeddings, trained_genre_embeddings, predicted_rating = tuned_model({
      "user_id": np.array([163298]),
      "title": np.array(['The Rescue']),
      "genres": np.array([get_genre('The Rescue')])
  })
print("Predicted rating:")
print(predicted_rating)

In [174]:
def extract_embeddings(model, unique_user_ids, unique_titles):
    """
    Extract embeddings for all users and movies using the trained model.
    
    Args:
    - model: The trained recommendation model.
    - unique_user_ids: List of unique user IDs.
    - unique_movie_ids: List of unique movie IDs.
    
    Returns:
    - user_embeddings: Embeddings for all users.
    - movie_embeddings: Embeddings for all movies.
    """
    # Extract movie embeddings
    titles = np.array(unique_titles)
    movie_embeddings = model.movie_model(tf.constant(titles, dtype=tf.string)).numpy()

    # Extract user embeddings
    user_ids = np.array(unique_user_ids)
    user_embeddings = model.user_model(tf.constant(user_ids, dtype=tf.int32)).numpy()

    return user_embeddings, movie_embeddings

In [175]:
def index_movie_embeddings(movie_embeddings):
    """
    Index the movie embeddings using FAISS.
    
    Args:
    - movie_embeddings: Embeddings for all movies.
    
    Returns:
    - index: FAISS index with movie embeddings.
    """
    # Dimension of the embeddings
    embedding_dimension = movie_embeddings.shape[1]

    # Create a FAISS index
    index = faiss.IndexFlatL2(embedding_dimension)

    # Add movie embeddings to the index
    index.add(movie_embeddings)

    return index

In [176]:
def recommend_movies(model, index, unique_titles, user_id, k=10):
    """
    Recommend movies for a given user by querying the FAISS index.
    
    Args:
    - model: The trained recommendation model.
    - index: FAISS index with movie embeddings.
    - unique_titles: List of unique movie titles.
    - user_id: The user ID for which to make recommendations.
    - k: Number of recommendations to retrieve (default is 10).
    
    Returns:
    - recommended_movie_ids: List of recommended movie IDs.
    """
    # Get the embedding for the given user
    user_embedding = model.user_model(tf.constant([user_id], dtype=tf.int32)).numpy()

    # Query the FAISS index
    distances, indices = index.search(user_embedding, k)

    # Get the recommended movie IDs
    recommended_movies = np.array(unique_titles)[indices[0]]

    return recommended_movies

In [177]:
# Extract embeddings
user_embeddings, movie_embeddings = extract_embeddings(tuned_model, unique_user_ids, unique_titles)

# Index movie embeddings
index = index_movie_embeddings(movie_embeddings)

# Example user ID for which to make recommendations
example_user_id = 56703

recommended_movie_ids = recommend_movies(tuned_model, index, unique_titles, example_user_id, k=3)

print(f"Recommended movie titles for user {example_user_id}: {recommended_movie_ids}")

Recommended movie titles for user 56703: [b'Goodbye, Petrushka' b'Jonas Brothers Family Roast' b'The Childe']


In [61]:
faiss.write_index(index, "trained_model/faiss/{}_faiss.index".format(timestamp))